# 📈 Stock Price LSTM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/stock-lstm-forecast/training/notebook.ipynb)

Forecast Apple's (AAPL) daily closing price. Four recurrent networks (SimpleRNN and LSTM on price levels,
an LSTM on **log-returns**, and a stacked multivariate LSTM with RSI/EMA indicators) are compared on the
original $ scale against the hardest baseline in finance, **naive persistence** ("tomorrow = today").
The best univariate model is exported with a frozen copy of the price history, so the
[Space](https://huggingface.co/spaces/shalev396/stock-lstm-forecast) can forecast from any cutoff date without calling a
market-data API.

All logic lives in `src/` (and `../model/model.py` for inference); this notebook calls it step by step.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "stock-lstm-forecast"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has; nothing platform-specific is installed or special-cased.
import tensorflow as tf
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("device:", device)

In [ ]:
# 1d. Project code: src/ (training) + ../model/model.py (architecture, preprocessing, Predictor)
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                   # ../model/model.py

print(utils.lib_versions("tensorflow", "keras", "numpy", "pandas", "sklearn"))
print("TensorFlow device:", utils.device_name("tensorflow"), "(float32 everywhere, no mixed precision)")

## 2. Config
Every hyperparameter lives in `src/config.py`: a 60-day window, a chronological 70/15/15 split, 64 units,
dropout 0.2, Adam 1e-3, batch 32, up to 40 epochs with early stopping (patience 6) on validation loss.
`SMOKE_TEST=1` keeps only the last 400 trading days, trains 8-unit models for one epoch, and exports to
`outputs/smoke/model/` instead of `../model/`.

In [ ]:
os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, "tensorflow")
print("export to:", cfg.model_dir.relative_to(utils.PROJECT_DIR))
cfg

## 3. Data
Daily AAPL OHLCV from 2015 via [yfinance](https://github.com/ranaroussi/yfinance), downloaded once into
`training/data/AAPL.csv` (a seeded synthetic random walk is the documented fallback when Yahoo is
unreachable; `data/DATA_SOURCE.txt` says which one you got). Prices are split-adjusted closes.

Three representations ("tracks") of the same days, all with scalers fit on the **train rows only**:
- `levels`: the close, MinMax-scaled;
- `returns`: daily log-returns `ln(C_t / C_(t-1))`, standardized with the train mean/std (the deployed one);
- `multivariate`: Close, Volume, RSI-14, EMA-20, EMA-50 (hand-rolled indicators), MinMax-scaled.

Each 60-day window belongs to the split of the day it predicts, so nothing crosses a split boundary.

In [ ]:
prices = data_setup.load_prices(cfg)
print(f"{len(prices):,} trading days, {prices.index[0].date()} -> {prices.index[-1].date()} |",
      "source:", data_setup.data_source())
tracks = data_setup.prepare_tracks(prices, cfg)
returns = tracks["returns"]
display(data_setup.split_summary(returns))
prices.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
bounds = {s: returns.dates[v[0]] for s, v in returns.idx.items()}
axes[0].plot(returns.dates, returns.close, color="#2a78d6", lw=1)
for split, color in (("val", "#eb6834"), ("test", "#1baf7a")):
    axes[0].axvline(bounds[split], color=color, ls="--", lw=1, label=f"{split} starts {bounds[split].date()}")
axes[0].set(title="AAPL close ($): train prices never reach the test range", ylabel="$")
axes[0].legend()
rets = np.diff(np.log(returns.close))
axes[1].hist(rets * 100, bins=120, color="#2a78d6")
axes[1].set(title=f"Daily log-returns: mean {rets.mean() * 100:.3f} %, std {rets.std() * 100:.2f} %", xlabel="%")
plt.show()

## 4. Load model
The deployed architecture comes from `../model/model.py` (`M.build_lstm`): a 60-step window of one
feature -> `LSTM(64)` -> `Dropout(0.2)` -> `Dense(1)`, float32. `model_builder` adds the comparison
variants (SimpleRNN, the same LSTM on levels, a stacked multivariate LSTM).

In [ ]:
variants = model_builder.build_variants(cfg, tracks)
variants["lstm_returns"].summary()
params = {name: utils.count_params(net) for name, net in variants.items()} | {model_builder.NAIVE: None}
pd.DataFrame({"track": model_builder.TRACKS, "params": params,
              "description": model_builder.describe(cfg)})

## 5. Training
Each variant is compiled (Adam + MSE on its scaled target) and fit with early stopping and
learning-rate halving on validation loss; the best-epoch weights are restored. About 2 minutes on a
desktop CPU for all four (TensorFlow uses a GPU automatically when one is visible).

In [ ]:
histories, start = {}, time.perf_counter()
for name, net in variants.items():
    engine.compile_model(net, cfg)
    histories[name] = engine.fit(net, tracks[model_builder.TRACKS[name]], cfg)
    h = histories[name]
    print(f"{name:20s} {len(h['loss']):2d} epochs (best {h['best_epoch']:2d}) "
          f"val_loss {min(h['val_loss']):.5f}  {h['seconds']:.1f}s")
train_time_s = time.perf_counter() - start
print(f"total {train_time_s:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, len(histories), figsize=(4 * len(histories), 3))
for ax, (name, h) in zip(np.atleast_1d(axes), histories.items()):
    ax.plot(h["loss"], label="train")
    ax.plot(h["val_loss"], label="validation")
    ax.set(title=name, xlabel="epoch", yscale="log")
np.atleast_1d(axes)[0].legend()
plt.tight_layout()
plt.show()

## 6. Evaluation
One-step-ahead predictions on the original $ scale for validation and test. The model is **selected on
validation RMSE** (among univariate models, which the Space can run recursively); the test split is
reported once. Then the deployed model is evaluated the way the Space uses it: **recursive multi-day
forecasts** from many test cutoffs, next to the naive "last close" forecast.

Expect the honest result: daily prices are close to a random walk, so the best a leak-free model can
do is roughly *match* persistence.

In [ ]:
evals = {name: engine.evaluate(net, tracks[model_builder.TRACKS[name]]) for name, net in variants.items()}
evals[model_builder.NAIVE] = engine.evaluate_naive(returns)
assert all(np.array_equal(tracks[t].idx["test"], returns.idx["test"]) for t in tracks)  # same test days
best = engine.select_best(evals, model_builder.DEPLOYABLE)
table = engine.comparison_table(evals, params)
print("deployed:", best, "| beats naive on test RMSE:", evals[best]["test"]["rmse"] < evals[model_builder.NAIVE]["test"]["rmse"])
table.round(4)

In [ ]:
horizon = engine.horizon_eval(variants[best], returns, cfg)
print(f"{horizon['n_origins']} test cutoffs ({horizon['first_origin']} -> {horizon['last_origin']}), "
      f"{horizon['horizon']}-day recursive forecasts: MAE {horizon['mae_mean']:.3f} $ vs naive {horizon['naive_mae_mean']:.3f} $")
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
idx = returns.idx["test"][-90:]
axes[0].plot(returns.dates[idx], returns.close[idx], color="black", label="actual")
axes[0].plot(returns.dates[idx], evals[best]["test_pred"][-90:], label=f"{best} (one-step)")
axes[0].plot(returns.dates[idx], evals[model_builder.NAIVE]["test_pred"][-90:], ls="--", label="naive")
axes[0].set(title="Last 90 test days", ylabel="$")
axes[0].legend()
steps = np.arange(1, horizon["horizon"] + 1)
axes[1].plot(steps, horizon["mae_by_step"], marker="o", label="LSTM recursive")
axes[1].plot(steps, horizon["naive_mae_by_step"], ls="--", label="naive (last close)")
axes[1].set(title="MAE by forecast step (test cutoffs)", xlabel="business days after the cutoff", ylabel="MAE ($)")
axes[1].legend()
plt.show()

## 7. Inference
Write the deployable files (`model.keras`, `config.json` and the frozen `prices.csv`) to `cfg.model_dir`,
then load them back with `model.load(dir, device).predict(cutoff, horizon)`: exactly what the Space and the
Inference Endpoint run. `save_model` also checks that the reloaded model reproduces the in-memory forecast.
The examples are the (cutoff, horizon) pairs in `../space/examples/cutoffs.csv`.

In [ ]:
export.save_model(cfg, variants[best], returns, prices, best, data_setup.data_source())
predictor = M.load(cfg.model_dir, "cpu")
print(f"frozen data: {predictor.prices.index[0].date()} -> {predictor.prices.index[-1].date()}, "
      f"valid cutoffs {predictor.first_cutoff.date()} -> {predictor.last_cutoff.date()}")
examples = pd.read_csv(utils.SPACE_DIR / "examples" / "cutoffs.csv")
fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 3.4))
for ax, row in zip(np.atleast_1d(axes), examples.itertuples()):
    res = predictor.predict(row.cutoff, row.horizon)
    print(f"{row.cutoff} +{row.horizon}d -> last history {res['history'][-1]}, forecast end {res['forecast'][-1]}, "
          f"mae {res['mae']}, naiveMae {res['naiveMae']}")
    for key, style in (("history", "-"), ("actual", "-"), ("forecast", "-"), ("naive", "--")):
        pts = pd.DataFrame(res[key])
        if len(pts):
            ax.plot(pd.to_datetime(pts["date"]), pts["price"], style, label=key)
    ax.set_title(f"cutoff {res['history'][-1]['date']}", fontsize=9)
    ax.tick_params(axis="x", labelrotation=30, labelsize=7)
np.atleast_1d(axes)[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

## 8. Export
Write `metrics.json` (deployed model, naive baseline, every variant, multi-step results), the card plots in
`assets/`, and refresh the model card's metrics and comparison tables. `cfg.model_dir` is then a complete
Hugging Face model repo. Set `PUSH_TO_HUB = True` in section 2 to upload it; the token comes from
`hf auth login` (local) or an `HF_TOKEN` secret / environment variable. Smoke runs are never uploaded.

In [ ]:
metrics = export.export_report(cfg, best=best, evals=evals, histories=histories, params=params,
                               horizon=horizon, tracks=tracks, dataset=f"AAPL daily closes ({data_setup.data_source()})",
                               device=utils.device_name("tensorflow"), train_time_s=train_time_s)
print({k: metrics[k] for k in ("model", "primary_metric", "baseline", "beats_naive", "params")})
for name in ("model_comparison", "test_forecast", "multi_step_mae", "training_curves"):
    display(Image(filename=str(cfg.model_dir / "assets" / f"{name}.png")))

In [ ]:
for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():30s} {path.stat().st_size / 1e3:9.1f} KB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))